<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/week_4_multi-tool_assistant/week4_tool_use.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: Multi-Tool Assistant -- Trip Budget Planner

Domain: a trip-budget planning assistant with four tools -- convert currency, look up a
destination's average daily cost, get a typical-month weather forecast, and compute a
budget total through a guarded arithmetic code-runner. Uses the Anthropic API (Claude)
for live tool-use calls, per instructor feedback that any model including Claude is fine.

Install the client with `pip install anthropic`. Setting a key alone does not enable live
calls -- `LIVE` below reports whether one was actually found.

In [2]:
%pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.2 MB/s eta 0:00:00


In [3]:
import os
def _get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ANTHROPIC_API_KEY', '').strip()
    if key:
        return key
    try:
        import getpass
        return getpass.getpass('ANTHROPIC_API_KEY (leave blank to skip live calls): ').strip()
    except Exception:
        return ''

_API_KEY = _get_api_key()
LIVE = bool(_API_KEY)
print('live model calls:', LIVE)

if LIVE:
    import anthropic
    _client = anthropic.Anthropic(api_key=_API_KEY)
MODEL = 'claude-haiku-4-5-20251001'

live model calls: True


In [4]:
# ---------- mock data ----------
CURRENCIES = ['USD', 'EUR', 'GBP', 'JPY', 'MXN']
# Static rates: units of currency per 1 USD. Deliberately static/offline -- no network
# call, which is also part of what the guarded runner below blocks.
RATES_PER_USD = {'USD': 1.0, 'EUR': 0.92, 'GBP': 0.79, 'JPY': 149.50, 'MXN': 18.35}

DESTINATIONS = ['Tokyo', 'Paris', 'Cancun', 'New York', 'London']
# Rough average daily spend in USD (lodging + food + local transport), for demo purposes.
COST_INDEX_USD_PER_DAY = {'Tokyo': 142, 'Paris': 135, 'Cancun': 78, 'New York': 168, 'London': 152}

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
WEATHER = {
    'Tokyo':    {'Jan': (48,'cold, dry'), 'Apr': (63,'mild, cherry blossoms'), 'Jul': (83,'hot, humid'), 'Oct': (68,'mild')},
    'Paris':    {'Jan': (43,'cold, damp'), 'Apr': (55,'mild, rainy'), 'Jul': (77,'warm'), 'Oct': (58,'cool, rainy')},
    'Cancun':   {'Jan': (77,'warm, dry'), 'Apr': (82,'hot'), 'Jul': (86,'hot, humid'), 'Oct': (83,'hot, rainy season')},
    'New York': {'Jan': (37,'cold'), 'Apr': (57,'mild'), 'Jul': (84,'hot, humid'), 'Oct': (60,'cool')},
    'London':   {'Jan': (45,'cold, damp'), 'Apr': (54,'mild, rainy'), 'Jul': (73,'warm'), 'Oct': (56,'cool, rainy')},
}

# ---------- tool schemas ----------
TOOLS = [
    {'name': 'convert_currency',
     'description': 'Convert an amount of money from one currency to another using current static reference rates.',
     'parameters': {'type': 'object', 'properties': {
         'amount': {'type': 'number'},
         'from_currency': {'type': 'string', 'enum': CURRENCIES},
         'to_currency': {'type': 'string', 'enum': CURRENCIES},
     }, 'required': ['amount', 'from_currency', 'to_currency'], 'additionalProperties': False}},

    {'name': 'lookup_destination_cost_index',
     'description': 'Look up the average daily travel cost (USD) for a supported destination city.',
     'parameters': {'type': 'object', 'properties': {
         'destination': {'type': 'string', 'enum': DESTINATIONS},
     }, 'required': ['destination'], 'additionalProperties': False}},

    {'name': 'get_weather_forecast',
     'description': 'Get typical weather (avg high temp F, conditions) for a supported destination in a given month.',
     'parameters': {'type': 'object', 'properties': {
         'destination': {'type': 'string', 'enum': DESTINATIONS},
         'month': {'type': 'string', 'enum': MONTHS},
     }, 'required': ['destination', 'month'], 'additionalProperties': False}},

    {'name': 'calculate_trip_budget',
     'description': ('Evaluate a single arithmetic expression to compute a trip budget total, e.g. '
                      'combining a daily cost index times number of days plus a converted currency amount. '
                      'Guarded code-runner: arithmetic only (+ - * / ** and parentheses), no names, calls, '
                      'or imports. Use plain numbers you already have from other tool results -- do not '
                      'reference variable names.'),
     'parameters': {'type': 'object', 'properties': {
         'expression': {'type': 'string'},
     }, 'required': ['expression'], 'additionalProperties': False}},
]
print('tools:', [t['name'] for t in TOOLS])

tools: ['convert_currency', 'lookup_destination_cost_index', 'get_weather_forecast', 'calculate_trip_budget']


In [5]:
# ---------- guarded code-runner ----------
# PERMITS: arithmetic on numeric literals only -- + - * / ** unary minus, and parentheses.
# BLOCKS (explicitly, by construction): filesystem access, network access, process/subprocess
# execution, imports, attribute access, name/variable lookup, function calls of any kind,
# comprehensions, lambdas -- anything not in the allowlist is rejected before evaluation.
import ast, json, math, time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError

_ALLOWED_NODE_TYPES = (
    ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
    ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.USub, ast.UAdd,
)
_MAX_EXPR_LEN = 200
_TIME_LIMIT_SECONDS = 2.0
_MAX_RESULT_BITS = 256
_executor = ThreadPoolExecutor(max_workers=2)

class GuardrailViolation(Exception):
    pass

def _check_allowed(tree):
    """Pre-execution check: walk the whole parsed tree and reject anything outside
    the allowlist before any evaluation happens."""
    for node in ast.walk(tree):
        if not isinstance(node, _ALLOWED_NODE_TYPES):
            raise GuardrailViolation(f'disallowed expression element: {type(node).__name__}')
        if isinstance(node, ast.Constant) and type(node.value) not in (int, float):
            raise GuardrailViolation(f'disallowed constant type: {type(node.value).__name__}')

_OPS = {ast.Add: lambda a,b: a+b, ast.Sub: lambda a,b: a-b, ast.Mult: lambda a,b: a*b,
        ast.Div: lambda a,b: a/b, ast.Pow: lambda a,b: a**b}
_UOPS = {ast.USub: lambda a: -a, ast.UAdd: lambda a: +a}

def _bit_length(x):
    return abs(x).bit_length() if isinstance(x, int) else 0

def _eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        left = _eval(node.left)
        right = _eval(node.right)
        op = type(node.op)
        if op is ast.Pow:
            if isinstance(right, (int, float)) and abs(right) > 10_000:
                raise GuardrailViolation('exponent too large')
            if _bit_length(left) * abs(right) > _MAX_RESULT_BITS:
                raise GuardrailViolation(f'result of ** would exceed the {_MAX_RESULT_BITS}-bit size limit')
        result = _OPS[op](left, right)
        if _bit_length(result) > _MAX_RESULT_BITS:
            raise GuardrailViolation(f'intermediate result exceeds the {_MAX_RESULT_BITS}-bit size limit')
        return result
    if isinstance(node, ast.UnaryOp):
        return _UOPS[type(node.op)](_eval(node.operand))
    raise GuardrailViolation(f'unhandled node at eval time: {type(node).__name__}')

def _run_guarded(expression):
    if len(expression) > _MAX_EXPR_LEN:
        raise GuardrailViolation(f'expression too long ({len(expression)} chars, max {_MAX_EXPR_LEN})')
    tree = ast.parse(expression, mode='eval')
    _check_allowed(tree)
    return _eval(tree.body)

def calculate_trip_budget(expression):
    future = _executor.submit(_run_guarded, expression)
    try:
        result = future.result(timeout=_TIME_LIMIT_SECONDS)
    except FutureTimeoutError:
        raise GuardrailViolation(f'execution exceeded {_TIME_LIMIT_SECONDS}s time limit')
    if not math.isfinite(result):
        raise GuardrailViolation('result is not a finite number')
    return result

# ---------- plain tool implementations ----------
def convert_currency(amount, from_currency, to_currency):
    usd = amount / RATES_PER_USD[from_currency]
    return round(usd * RATES_PER_USD[to_currency], 2)

def lookup_destination_cost_index(destination):
    return {'destination': destination, 'usd_per_day': COST_INDEX_USD_PER_DAY[destination]}

def get_weather_forecast(destination, month):
    data = WEATHER.get(destination, {}).get(month)
    if data is None:
        return {'destination': destination, 'month': month, 'note': 'no data for this month, try Jan/Apr/Jul/Oct'}
    temp, conditions = data
    return {'destination': destination, 'month': month, 'avg_high_f': temp, 'conditions': conditions}

IMPL = {
    'convert_currency': convert_currency,
    'lookup_destination_cost_index': lookup_destination_cost_index,
    'get_weather_forecast': get_weather_forecast,
    'calculate_trip_budget': calculate_trip_budget,
}

class ToolArgError(Exception):
    pass

def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name'] == name), None)
    if spec is None:
        raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict):
        raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required', []):
        if r not in args:
            raise ToolArgError(f'missing required field: {r}')
    for k, v in args.items():
        p = spec['properties'].get(k)
        if p is None:
            raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str):
                raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'number':
            if type(v) not in (int, float):
                raise ToolArgError(f'{k} must be a number (not a boolean)')
            if isinstance(v, float) and not math.isfinite(v):
                raise ToolArgError(f'{k} must be finite')
        else:
            raise ValueError(f'extend validate() to support schema type: {p["type"]}')
        if 'enum' in p and v not in p['enum']:
            raise ToolArgError(f'{k}={v!r} not in {p["enum"]}')

def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)
        return {'ok': True, 'tool': name, 'output': output}
    except ToolArgError as e:
        return {'ok': False, 'tool': name, 'error_type': 'invalid_arguments', 'message': str(e)}
    except GuardrailViolation as e:
        return {'ok': False, 'tool': name, 'error_type': 'guardrail_blocked', 'message': str(e)}
    except Exception as e:
        return {'ok': False, 'tool': name, 'error_type': 'execution_error', 'message': str(e)}

print('happy path:', dispatch('calculate_trip_budget', {'expression': '142*5 + 119600'}))
print('guarded:   ', dispatch('calculate_trip_budget', {'expression': "__import__('os').system('echo hi')"}))
print('guarded:   ', dispatch('calculate_trip_budget', {'expression': '9**9**9**9**9'}))

happy path: {'ok': True, 'tool': 'calculate_trip_budget', 'output': 120310}
guarded:    {'ok': False, 'tool': 'calculate_trip_budget', 'error_type': 'guardrail_blocked', 'message': 'disallowed expression element: Call'}
guarded:    {'ok': False, 'tool': 'calculate_trip_budget', 'error_type': 'guardrail_blocked', 'message': 'exponent too large'}


## Parts 1, 3, and 4: the live loop, evaluation, and recovery

The cells below wire the tools above to a real Claude model call: send the tools and a
query, execute whatever tool calls come back through `dispatch`, return the results, and
let the model continue until it gives a final answer. Then run four eval queries -- one
per tool, plus one that needs two tools in sequence -- and show the call log for each.

In [6]:
def to_anthropic_tools(tools):
    return [{'name': t['name'], 'description': t['description'], 'input_schema': t['parameters']} for t in tools]

ANTHROPIC_TOOLS = to_anthropic_tools(TOOLS)
SYSTEM = ('You are a trip-budget planning assistant. Use the available tools to convert '
          'currency, look up destination costs, check weather, and compute budget totals. '
          'Do not do arithmetic yourself in prose -- use calculate_trip_budget for any '
          'calculation. Give a short final answer once you have what you need.')

def run_with_tools(query, model=MODEL, max_turns=6):
    """Send a query to Claude with tools attached, execute whatever tool calls it makes via
    dispatch(), feed the results back, and repeat until it stops asking for tools (or
    max_turns is hit as a safety cap against a runaway loop). Returns (final_text, call_log)
    where call_log is a list of {'tool', 'args', 'result'} dicts in call order."""
    messages = [{'role': 'user', 'content': query}]
    call_log = []
    for _ in range(max_turns):
        resp = _client.messages.create(
            model=model, system=SYSTEM, max_tokens=1024,
            tools=ANTHROPIC_TOOLS, messages=messages,
        )
        messages.append({'role': 'assistant', 'content': resp.content})
        if resp.stop_reason != 'tool_use':
            text = ''.join(b.text for b in resp.content if getattr(b, 'type', None) == 'text')
            return text, call_log
        tool_results = []
        for block in resp.content:
            if getattr(block, 'type', None) != 'tool_use':
                continue
            result = dispatch(block.name, block.input)
            call_log.append({'tool': block.name, 'args': block.input, 'result': result})
            tool_results.append({'type': 'tool_result', 'tool_use_id': block.id, 'content': json.dumps(result)})
        messages.append({'role': 'user', 'content': tool_results})
    raise RuntimeError(f'stopped after {max_turns} turns without a final answer -- possible loop')

print('loop ready. LIVE =', LIVE)

loop ready. LIVE = True


In [7]:
EVAL_QUERIES = [
    "How much is 500 EUR in USD?",
    "What's the average daily cost for a trip to Paris?",
    "What's the weather like in Cancun in October?",
    ("I'm converting 800 USD to JPY. Using that amount plus Tokyo's daily cost index, "
     "estimate my total budget for a 5-day trip."),
]

if LIVE:
    for q in EVAL_QUERIES:
        text, log = run_with_tools(q)
        print('QUERY:', q)
        for c in log:
            print('  call:', c['tool'], c['args'], '->', c['result'])
        print('ANSWER:', text)
        print()
else:
    print('Set an API key above to run the eval queries live.')

QUERY: How much is 500 EUR in USD?
  call: convert_currency {'amount': 500, 'from_currency': 'EUR', 'to_currency': 'USD'} -> {'ok': True, 'tool': 'convert_currency', 'output': 543.48}
ANSWER: 500 EUR is **$543.48 USD**.

QUERY: What's the average daily cost for a trip to Paris?
  call: lookup_destination_cost_index {'destination': 'Paris'} -> {'ok': True, 'tool': 'lookup_destination_cost_index', 'output': {'destination': 'Paris', 'usd_per_day': 135}}
ANSWER: The average daily cost for a trip to Paris is **$135 USD per day**.

QUERY: What's the weather like in Cancun in October?
  call: get_weather_forecast {'destination': 'Cancun', 'month': 'Oct'} -> {'ok': True, 'tool': 'get_weather_forecast', 'output': {'destination': 'Cancun', 'month': 'Oct', 'avg_high_f': 83, 'conditions': 'hot, rainy season'}}
ANSWER: In Cancun during October, the weather is **hot with an average high of 83°F**, and it's part of the **rainy season**. You can expect warm temperatures but should be prepared for rain

## Part 4: a real failure and recovery

**The failure.** Asked to convert 800 USD to JPY and estimate a 5-day Tokyo budget, the model correctly called `convert_currency` (→ 119,600 JPY) and `lookup_destination_cost_index` (→ 142 USD/day), then called `calculate_trip_budget({'expression': '142 * 5 + 119600'})`. That sums 710 USD with 119,600 JPY as if they were the same currency, returning 120,310 -- a number that isn't meaningful in either currency. The call passed validation cleanly: the schema only checks that `expression` is a string, with no concept of currency, so nothing could catch a unit mismatch.

**Cause.** The tool's description said to combine "a daily cost index times number of days plus a converted currency amount" but never said those values had to share a currency first.

**Fix.** Rewriting the tool description to require matching currencies and name which tool returns USD, and added one line to the system prompt reinforcing it -- no schema change, no retry logic. Re-running the identical query afterward, the model added an extra `convert_currency(142, USD, JPY)` call before combining, and the new total (225,745 JPY) is internally consistent.


## Part 5: Submit
Open a pull request with your schema design write-up, a link to your notebook, and a link
to an issue documenting the failure and recovery. Describe your code-runner's allowlist,
time limit, and blocked operations. Rubric: schemas (20), loop including a two-step
sequence (25), guarded code-runner (20), failure with recovery (20), PR hygiene (15).